# Wheel-Constrained Matrix Simulator and Trajectory Control

This notebook develops a step-through simulator for the project’s four-wheel omni robot, a conventional PID baseline, and a matching convex MPC QP. The reference is time-indexed position/velocity/acceleration (plus heading), not a single endpoint, so the controller plans through turns instead of demanding full throttle toward a goal.

## Model boundary

- Physical simulator state is `[x_world, y_world, phi, vx_world, vy_world, omega]`; the state derivative uses the same world-frame convention. Body-frame quantities are formed explicitly for wheel dynamics.
- PID integral memory belongs to the PID controller. MPC predicted integral errors are auxiliary optimization/controller variables, not plant states.
- In this first plant, rolling is a hard kinematic constraint. Wheel speed and acceleration are derived from the body twist; wheel inertia is not dropped. It enters the coupled effective inertia and the wheel torque residual is checked every step.
- The actuator source uses `J_w*theta_ddot = K_t*i + tau_load_signed`; `simulation.py` forms the signed load torque as `R*F_d`. If load torque is instead stored as a positive resisting magnitude, use `J_w*theta_ddot = K_t*i - tau_load_magnitude`.
- The rolling constraint is allowed only while required traction satisfies `abs(F_d,i) <= MU_D_STAT * F_N`. If that set is infeasible, the rolling plant is invalid for that command; later sliding dynamics need actual wheel-speed states and kinetic friction.
- Each MPC solve uses a preselected convex safe corridor: static and predicted dynamic obstacles contribute fixed-side linear half-space constraints at each horizon stage. Obstacle-side selection happens outside the QP; a disjunction of sides is not included in one convex solve. The PID baseline uses repulsive acceleration, and every plant step records a post-step clearance check.
- Heading remains a tracked state. Use the path-tangent callback for transit or `target_heading_reference(target_world)` to face a pass/shot target; changing heading is coupled to wheel currents through the locally linearized model.
- The passive roller axial force is assumed zero here as an ideal-free-roller approximation, not claimed to be physically nonexistent. Disturbance force/torque is a separate body-frame callback.
- Current is the controller input and `IQ_LIMIT` is imported from the constants module. The `4 m/s` speed cap and `3 rad/s` yaw-rate cap are provisional study limits. `DT=0.01 s` is a starting controller sample, not yet certified by convergence or hardware timing tests.

## Reference documents

The local `archive/Project.pdf` is an architecture sketch: reference generation, controller, wheel commands/forces, robot dynamics, integration, then logging. It explicitly calls for wheel-direction and symmetry checks.

`COCO_Project_Report1_anonymized.pdf` is a drone-racing RTI/SQP NMPC example, not this robot's physics model. Relevant transferable ideas are stage-wise trajectory tracking, explicit state/input constraints, warm-started receding-horizon solves, and respecting solver latency. Its dynamics and SQP/RTI implementation do not directly establish convexity for this wheel model. This notebook instead uses a locally linearized, time-varying model and a convex QP, and tracks each future reference sample rather than repeatedly aiming at one horizon-end point.

Project constants such as `MU_D_STAT` remain provisional measurements/assumptions; the friction envelope is a design constraint, not validation that the robot never slips.

## 1. Import Required Libraries

Import the project’s robot constants and the Python libraries used for simulation and plotting. NumPy and Matplotlib are required; CVXPY is used in the later optional QP section.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import importlib
import robot_constants as _robot_constants

# Refresh cached project constants when this setup cell is rerun.
_robot_constants = importlib.reload(_robot_constants)
from robot_constants import (
    ALPHA_A,
    F_N,
    IQ_LIMIT,
    K_SMOOTH_A,
    MOTOR_KT,
    MU_A_DYN,
    MU_A_STAT,
    MU_D_STAT,
    ROBOT_INERTIA,
    ROBOT_MASS,
    WHEEL_ALPHA_DEG,
    WHEEL_INERTIA,
    WHEEL_RADIUS,
    WHEEL_ROBOT_DIST,
)

CONTROLLER_MODE = "PID"  # Keep MPC paused while validating the plant model.
MPC_HORIZON = 8           # Current feasibility-tested LTV prediction horizon.
MPC_CONSTRAINT_TOL = 1e-4  # Match OSQP absolute tolerance when reporting feasibility.
DT = 0.01
MAX_ACCELERATION = 5.5  # m/s^2; below the modeled current/traction acceleration capability
MAX_VELOCITY = 4.5      # m/s; fast soccer-robot transit target, still actuator-limited
MAX_YAW_RATE = 3.0      # rad/s; provisional until measured
N_WHEELS = len(WHEEL_ALPHA_DEG)
# MU_D_STAT limits drive traction; F_N is calculated separately as robot weight divided across the wheels.
F_ROLL_LIMIT = MU_D_STAT * F_N

## 2. Reference Targets: Position, Velocity, and Heading at Each Time

This notebook does not ask the MPC to invent a route. `ROUTE_WAYPOINTS` is a manually selected centerline that already passes on chosen sides of the static obstacles. The code parameterizes that route by time and supplies a desired position and velocity at every MPC stage. These references are fixed data; the QP chooses currents and predicted robot states to track them.

The route's forward/backward speed envelope accelerates from rest, stays under configured speed and acceleration limits, and brakes to rest at the last waypoint. That makes the endpoint a receiving-pass style target: goal position with zero translational velocity. The path itself is not the convexity claim.

Convexity comes from the optimization problem: at each horizon stage the obstacle side is fixed from the nominal reference, so obstacle avoidance is a linear half-space. State updates are affine and the tracking/effort costs are quadratic. The QP cannot decide to pass either side of an obstacle in the same solve; changing sides is an outer route-choice decision. The drone report uses a fixed spline and nonlinear SQP/RTI tracking, and says trajectory planning is outside its scope.

Heading is another time-indexed target. Use the route tangent during transit, a fixed target point for pass/shot behavior, or provide a custom callback for a scheduled heading. For a receiving pass, set the endpoint velocity to zero and choose the heading toward the passer or intended shot target.


In [124]:
from scipy.interpolate import CubicSpline

# Four control points shape a smooth route that fits inside the 4.5 m x 6 m
# half-field. The route bends above the obstacles below; MPC tracks these
# waypoints and does not choose a new global path inside its convex QP.
FIELD_LENGTH = 4.5  # x direction [m]
FIELD_WIDTH = 6.0   # y direction [m]
ROUTE_CONTROL_POINTS = np.array([
    [-2.02, -2.65],  # start, with room to the field boundary
    [-1.42, -0.30],  # lift the path toward the upper side of the first obstacle
    [-0.20,  3.00],  # pull the broad arc across both direct-line obstacles
    [ 2.02,  2.65],  # final target, inside the 4.5 x 6 m field
])

def cubic_bezier_points(control_points, samples=41):
    """Sample a cubic Bezier centerline to create smooth, human-readable waypoints."""
    t = np.linspace(0.0, 1.0, samples)[:, None]
    return ((1 - t)**3 * control_points[0]
            + 3 * (1 - t)**2 * t * control_points[1]
            + 3 * (1 - t) * t**2 * control_points[2]
            + t**3 * control_points[3])

ROUTE_WAYPOINTS = cubic_bezier_points(ROUTE_CONTROL_POINTS)
# These static obstacles block the direct start-to-goal chord. The chosen route
# intentionally passes above both, with the robot-clearance margin in the MPC.
STATIC_OBSTACLES = [
    np.array([[-0.42, -0.40], [0.42, -0.40], [0.42, 0.40], [-0.42, 0.40]]),
    np.array([[0.55, 0.65], [1.25, 0.65], [1.25, 1.25], [0.55, 1.25]]),
]
# Each dynamic obstacle stores polygon vertices in its local frame and a function
# returning its predicted world-frame translation at time_s. Example:
# {"vertices": local_rectangle, "position_fn": lambda time_s: np.array([x0+vx*time_s, y0+vy*time_s])}
DYNAMIC_OBSTACLES = []
ROBOT_CLEARANCE = 0.35  # m; footprint + safety margin + prediction/discretization margin
# Leave a small margin under MAX_ACCELERATION when combining turns with
# speeding up or braking along the route.
LATERAL_ACCEL_FRACTION = 0.90
TANGENTIAL_ACCEL_FRACTION = 0.85


def wheel_directions_body():
    """Return drive and axle unit vectors as columns, ordered by WHEEL_ALPHA_DEG.

    alpha_i is measured from world/body +x counterclockwise. The project uses
    a_hat=[cos(alpha), sin(alpha)] and d_hat=[-sin(alpha), cos(alpha)].
    """
    alpha = np.radians(np.asarray(WHEEL_ALPHA_DEG, dtype=float))
    return np.vstack((-np.sin(alpha), np.cos(alpha))), np.vstack((np.cos(alpha), np.sin(alpha)))


def drive_slip_velocity(velocity_body, theta_dot_actual):
    """Return Williams d-axis contact slip for actual wheel speeds [rad/s]."""
    drive, _ = wheel_directions_body()
    return drive.T @ np.asarray(velocity_body, dtype=float) + WHEEL_RADIUS * np.asarray(theta_dot_actual, dtype=float)


def axial_slip_velocity_body(velocity_body, omega):
    """Compute v_slip,a = V_center dot a_hat in the body frame [m/s].

    For wheel i, V_center = V_body + omega x r_i, with
    omega x r_i = [-omega*r_y, omega*r_x]. Wheel spin adds only along d_hat,
    and d_hat dot a_hat=0, so it does not enter this axial projection.
    """
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    wheel_center_velocity = np.asarray(velocity_body, dtype=float)[:, None] + np.vstack((
        -omega * radius[1], omega * radius[0],
    ))
    return np.sum(axle * wheel_center_velocity, axis=0)


def mu_a_from_slip(slip_a):
    """Use the project's provisional static/dynamic axial friction interpolation."""
    return MU_A_DYN + (MU_A_STAT - MU_A_DYN) * np.exp(-ALPHA_A * np.abs(slip_a))


def axial_friction_wrench_body(velocity_body, omega):
    """Compute passive-roller force from Williams-style smooth Coulomb friction.

    F_a,i = -F_N * mu_a(v_slip,a,i) * (2/pi) * atan(K_SMOOTH_A*v_slip,a,i).
    Return net body wrench [Fx,Fy,tau_z], per-wheel slip, and per-wheel force.
    """
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    slip_a = axial_slip_velocity_body(velocity_body, omega)
    mu_a = mu_a_from_slip(slip_a)
    force_a = -F_N * mu_a * (2.0 / np.pi) * np.arctan(K_SMOOTH_A * slip_a)
    force_vectors = axle * force_a
    total_force = np.sum(force_vectors, axis=1)
    total_torque = np.sum(radius[0] * force_vectors[1] - radius[1] * force_vectors[0])
    return np.r_[total_force, total_torque], slip_a, force_a


def convex_polygon_halfspaces(vertices):
    """Return unit outward edge normals n and bounds b for a CCW convex polygon."""
    vertices = np.asarray(vertices, dtype=float)
    edges = np.roll(vertices, -1, axis=0) - vertices
    normals = np.column_stack((edges[:, 1], -edges[:, 0]))
    normals /= np.linalg.norm(normals, axis=1, keepdims=True)
    return normals, np.sum(normals * vertices, axis=1)


def obstacle_safe_halfspace(route_point, vertices, clearance=ROBOT_CLEARANCE):
    """Choose the locally safe side of a convex obstacle as n dot p >= b+clearance.

    The side is selected from the route point and held fixed in the local QP.
    Obstacle-side selection itself is discrete/nonconvex; the resulting halfspace
    is affine and convex for that selected route corridor.
    """
    route_point = np.asarray(route_point, dtype=float)
    vertices = np.asarray(vertices, dtype=float)
    normals, _ = convex_polygon_halfspaces(vertices)
    closest_point = None
    edge_normal = None
    min_distance = np.inf
    for edge_index, start in enumerate(vertices):
        end = vertices[(edge_index + 1) % len(vertices)]
        edge = end - start
        fraction = np.clip(np.dot(route_point - start, edge) / np.dot(edge, edge), 0.0, 1.0)
        candidate = start + fraction * edge
        distance = np.linalg.norm(route_point - candidate)
        if distance < min_distance:
            min_distance = distance
            closest_point = candidate
            edge_normal = normals[edge_index]
    normal = edge_normal if min_distance < 1e-10 else (route_point - closest_point) / min_distance
    return normal, float(normal @ closest_point + clearance)


def point_polygon_distance(point, vertices):
    """Return Euclidean clearance from a point to a convex polygon boundary/interior."""
    point = np.asarray(point, dtype=float)
    vertices = np.asarray(vertices, dtype=float)
    normals, bounds = convex_polygon_halfspaces(vertices)
    if np.all(normals @ point <= bounds):
        return 0.0
    distances = []
    for edge_index, start in enumerate(vertices):
        edge = vertices[(edge_index + 1) % len(vertices)] - start
        fraction = np.clip(np.dot(point - start, edge) / np.dot(edge, edge), 0.0, 1.0)
        distances.append(np.linalg.norm(point - (start + fraction * edge)))
    return float(min(distances))


def obstacle_polygons_at(time_s):
    """Return named world-frame obstacle polygons and required clearances at time_s."""
    obstacles = [
        (f"static_{index}", np.asarray(vertices, dtype=float), ROBOT_CLEARANCE)
        for index, vertices in enumerate(STATIC_OBSTACLES)
    ]
    for index, obstacle in enumerate(DYNAMIC_OBSTACLES):
        translation = np.asarray(obstacle["position_fn"](time_s), dtype=float)
        if translation.shape != (2,):
            raise ValueError("dynamic obstacle position_fn must return [x, y]")
        vertices = np.asarray(obstacle["vertices"], dtype=float) + translation
        obstacles.append((
            obstacle.get("name", f"dynamic_{index}"), vertices,
            float(obstacle.get("clearance", ROBOT_CLEARANCE)),
        ))
    return obstacles


def obstacle_clearance_report(position, time_s):
    """Measure center-to-polygon clearance after a plant step; negative margin is collision."""
    report = []
    for name, vertices, required_clearance in obstacle_polygons_at(time_s):
        distance = point_polygon_distance(position, vertices)
        report.append({
            "name": name,
            "distance_m": distance,
            "required_clearance_m": required_clearance,
            "margin_m": distance - required_clearance,
        })
    return report


def pid_obstacle_repulsion(position, time_s, gain=4.0, influence_distance=1.0):
    """Add a smooth, local repulsive acceleration for the PID comparison baseline."""
    acceleration = np.zeros(2)
    for _, vertices, required_clearance in obstacle_polygons_at(time_s):
        distance = point_polygon_distance(position, vertices)
        gap = distance - required_clearance
        if gap < influence_distance:
            normal, _ = obstacle_safe_halfspace(position, vertices, clearance=0.0)
            acceleration += gain * (influence_distance - max(gap, 0.0)) * normal
    return acceleration


# Fit the centerline by chord length, then reparameterize samples by measured arc length.
_route_chords = np.linalg.norm(np.diff(ROUTE_WAYPOINTS, axis=0), axis=1)
_route_knots = np.r_[0.0, np.cumsum(_route_chords)]
_route_x = CubicSpline(_route_knots, ROUTE_WAYPOINTS[:, 0], bc_type="natural")
_route_y = CubicSpline(_route_knots, ROUTE_WAYPOINTS[:, 1], bc_type="natural")
_route_parameter = np.linspace(0.0, _route_knots[-1], 5001)
_route_points = np.column_stack((_route_x(_route_parameter), _route_y(_route_parameter)))
PATH_ARC = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(_route_points, axis=0), axis=1))]
_route_dx = _route_x(_route_parameter, 1)
_route_dy = _route_y(_route_parameter, 1)
_route_ddx = _route_x(_route_parameter, 2)
_route_ddy = _route_y(_route_parameter, 2)
_route_tangent_norm = np.maximum(np.hypot(_route_dx, _route_dy), 1e-9)
PATH_TANGENT = np.column_stack((_route_dx / _route_tangent_norm, _route_dy / _route_tangent_norm))
PATH_CURVATURE = (_route_dx * _route_ddy - _route_dy * _route_ddx) / np.maximum(_route_tangent_norm**3, 1e-9)

# Build a fast reference profile before the MPC runs: curvature limits corner
# speed, a forward pass limits acceleration, and a backward pass reserves room
# to brake to zero at the final waypoint.
LATERAL_ACCEL_LIMIT = LATERAL_ACCEL_FRACTION * MAX_ACCELERATION
TANGENTIAL_ACCEL_LIMIT = TANGENTIAL_ACCEL_FRACTION * MAX_ACCELERATION
PATH_SPEED_CAP = np.minimum(
    MAX_VELOCITY,
    np.sqrt(LATERAL_ACCEL_LIMIT / np.maximum(np.abs(PATH_CURVATURE), 1e-6)),
)
PATH_SPEED = PATH_SPEED_CAP.copy()
PATH_SPEED[0] = 0.0
PATH_SPEED[-1] = 0.0
PATH_DS = np.diff(PATH_ARC)
for _ in range(2):
    for index in range(1, len(PATH_SPEED)):
        PATH_SPEED[index] = min(
            PATH_SPEED[index],
            np.sqrt(max(0.0, PATH_SPEED[index - 1]**2 + 2.0 * TANGENTIAL_ACCEL_LIMIT * PATH_DS[index - 1])),
        )
    for index in range(len(PATH_SPEED) - 2, -1, -1):
        PATH_SPEED[index] = min(
            PATH_SPEED[index],
            np.sqrt(max(0.0, PATH_SPEED[index + 1]**2 + 2.0 * TANGENTIAL_ACCEL_LIMIT * PATH_DS[index])),
        )
# Approximate each interval travel time from its arc length and endpoint speeds.
# PATH_TIME maps route distance onto clock time for the controller references.
PATH_DT = 2.0 * PATH_DS / np.maximum(PATH_SPEED[:-1] + PATH_SPEED[1:], 1e-6)
PATH_TIME = np.r_[0.0, np.cumsum(PATH_DT)]
PATH_END_TIME = float(PATH_TIME[-1])
PATH_TANGENTIAL_ACCEL = np.gradient(PATH_SPEED**2 / 2.0, PATH_ARC)
PATH_NORMAL = np.column_stack((-PATH_TANGENT[:, 1], PATH_TANGENT[:, 0]))


def trajectory_reference(time_s):
    """Return desired world position, velocity, and acceleration at time_s.

    These are fixed references, not optimization variables. The MPC tracks
    them; after PATH_END_TIME the desired robot remains at the goal and stopped.
    """
    if time_s <= 0.0:
        arc_position, speed, tangential_acceleration = PATH_ARC[0], PATH_SPEED[0], PATH_TANGENTIAL_ACCEL[0]
    elif time_s >= PATH_END_TIME:
        # Receiving-pass target: stay at the goal with zero planar velocity.
        return ROUTE_WAYPOINTS[-1].copy(), np.zeros(2), np.zeros(2)
    else:
        arc_position = np.interp(time_s, PATH_TIME, PATH_ARC)
        speed = np.interp(arc_position, PATH_ARC, PATH_SPEED)
        tangential_acceleration = np.interp(arc_position, PATH_ARC, PATH_TANGENTIAL_ACCEL)

    position = np.array([
        np.interp(arc_position, PATH_ARC, _route_points[:, 0]),
        np.interp(arc_position, PATH_ARC, _route_points[:, 1]),
    ])
    tangent = np.array([
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 0]),
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 1]),
    ])
    tangent /= max(np.linalg.norm(tangent), 1e-9)
    normal = np.array([-tangent[1], tangent[0]])
    curvature = np.interp(arc_position, PATH_ARC, PATH_CURVATURE)
    velocity = speed * tangent
    acceleration = tangential_acceleration * tangent + speed**2 * curvature * normal
    return position, velocity, acceleration


def body_to_world(vector_body, phi):
    """Rotate a body-frame vector into the world frame."""
    cosine, sine = np.cos(phi), np.sin(phi)
    return np.array([[cosine, -sine], [sine, cosine]]) @ np.asarray(vector_body, dtype=float)


def world_to_body(vector_world, phi):
    """Rotate a world-frame vector into the body frame, matching simulation.py."""
    cosine, sine = np.cos(phi), np.sin(phi)
    return np.array([[cosine, sine], [-sine, cosine]]) @ np.asarray(vector_world, dtype=float)


def wrap_angle(angle):
    return (angle + np.pi) % (2.0 * np.pi) - np.pi


def tangent_heading_reference(time_s, position, velocity):
    """Face along the route tangent, including when the target speed is zero.

    Heading cannot be recovered from velocity at rest: the velocity direction is
    undefined there. Instead, read the geometric route tangent at this reference time.
    """
    # Clamp before/after the route so the robot holds the first/last route heading.
    clamped_time = np.clip(time_s, 0.0, PATH_END_TIME)
    arc_position = np.interp(clamped_time, PATH_TIME, PATH_ARC)
    tangent = np.array([
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 0]),
        np.interp(arc_position, PATH_ARC, PATH_TANGENT[:, 1]),
    ])
    return float(np.arctan2(tangent[1], tangent[0]))


def target_heading_reference(target_world):
    """Create a heading callback that faces a fixed pass/shot target.

    At every time, the desired angle points from the robot position to target_world.
    """
    target_world = np.asarray(target_world, dtype=float)

    def heading(time_s, position, velocity):
        direction = target_world - position
        return float(np.arctan2(direction[1], direction[0]))

    return heading


HEADING_MODE = "path"  # Set to "target" to face a pass/shot target.
HEADING_TARGET_WORLD = np.array([5.0, 0.0])


def configured_heading_reference():
    """Build the selected heading reference used by PID and MPC."""
    if HEADING_MODE.lower() == "path":
        return tangent_heading_reference
    if HEADING_MODE.lower() == "target":
        return target_heading_reference(HEADING_TARGET_WORLD)
    raise ValueError("HEADING_MODE must be 'path' or 'target'")


def wheel_directions_body():
    """Return drive and axle unit vectors as columns in wheel-constant order."""
    alpha = np.radians(np.asarray(WHEEL_ALPHA_DEG, dtype=float))
    return np.vstack((-np.sin(alpha), np.cos(alpha))), np.vstack((np.cos(alpha), np.sin(alpha)))


def drive_slip_velocity(velocity_body, theta_dot_actual):
    """Compute Williams d-axis contact slip from actual wheel speed [rad/s]."""
    drive, _ = wheel_directions_body()
    return drive.T @ np.asarray(velocity_body, dtype=float) + WHEEL_RADIUS * np.asarray(theta_dot_actual, dtype=float)


def axial_slip_velocity_body(velocity_body, omega):
    """Compute v_slip,a = V_center dot a_hat in the body frame [m/s]."""
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    wheel_center_velocity = np.asarray(velocity_body, dtype=float)[:, None] + np.vstack((
        -omega * radius[1], omega * radius[0],
    ))
    return np.sum(axle * wheel_center_velocity, axis=0)


def mu_a_from_slip(slip_a):
    """Use the project axial friction coefficient interpolation, documented as provisional."""
    return MU_A_DYN + (MU_A_STAT - MU_A_DYN) * np.exp(-ALPHA_A * np.abs(slip_a))


def axial_friction_wrench_body(velocity_body, omega):
    """Compute smooth Coulomb passive-roller force and moment from each a-axis slip."""
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    slip_a = axial_slip_velocity_body(velocity_body, omega)
    force_a = -F_N * mu_a_from_slip(slip_a) * (2.0 / np.pi) * np.arctan(K_SMOOTH_A * slip_a)
    force_vectors = axle * force_a
    total_force = np.sum(force_vectors, axis=1)
    total_torque = np.sum(radius[0] * force_vectors[1] - radius[1] * force_vectors[0])
    return np.r_[total_force, total_torque], slip_a, force_a


def zero_disturbance(time_s, physical_state=None):
    """Return no external body-frame wrench [Fx, Fy, torque]."""
    return np.zeros(3)


def sinusoidal_force_disturbance(amplitude=(0.0, 0.0), frequency=1.0, torque_amplitude=0.0):
    """Create a time-varying body-frame wrench callback for disturbance tests."""
    amplitude = np.asarray(amplitude, dtype=float)
    if amplitude.shape != (2,):
        raise ValueError("amplitude must be [Fx, Fy]")

    def disturbance(time_s, physical_state=None):
        phase = 2.0 * np.pi * frequency * time_s
        return np.r_[amplitude * np.sin(phase), torque_amplitude * np.sin(phase)]

    return disturbance

In [125]:
def axial_slip_velocity_body(velocity_body, omega):
    """Match williams_model.slip_velocities()'s contact velocity projected on each axle."""
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    wheel_center_velocity = np.asarray(velocity_body, dtype=float)[:, None] + np.vstack((
        -omega * radius[1],
        omega * radius[0],
    ))
    return np.sum(axle * wheel_center_velocity, axis=0)


def mu_a_from_slip(slip_a):
    """Project axial friction coefficient interpolation from robot_constants.py."""
    return MU_A_DYN + (MU_A_STAT - MU_A_DYN) * np.exp(-ALPHA_A * np.abs(slip_a))


def axial_friction_wrench_body(velocity_body, omega):
    """Return net passive-roller a-axis body wrench, slips, and per-wheel friction forces."""
    _, axle = wheel_directions_body()
    radius = WHEEL_ROBOT_DIST * axle
    slip_a = axial_slip_velocity_body(velocity_body, omega)
    force_a = -F_N * mu_a_from_slip(slip_a) * (2.0 / np.pi) * np.arctan(K_SMOOTH_A * slip_a)
    force_vectors = axle * force_a
    total_force = np.sum(force_vectors, axis=1)
    total_torque = np.sum(radius[0] * force_vectors[1] - radius[1] * force_vectors[0])
    return np.r_[total_force, total_torque], slip_a, force_a

## 3. Implement the Robot Kinematic and Matrix Model

The first plant is fixed-heading planar translation. Wheel drive directions are taken from `WHEEL_ALPHA_DEG`; yaw and axial roller resistance are intentionally left out. The class exposes the discrete model `A`, `B`, and reference offset at each step, so a caller can inspect the exact predictor before advancing the plant.

In [126]:
class RollingOnlyRobotSimulator:
    """Rolling-only plant using the same force balance as the old simulator.

    State order is [x_world, y_world, phi, vx_world, vy_world, omega].
    The control is one torque-producing current per wheel. Drive slip is not
    integrated here: wheel speeds follow the no-slip rolling constraint.
    """

    def __init__(self, dt=DT, initial_state=None, disturbance_fn=zero_disturbance,
                 disturbance_forecast=None, heading_reference_fn=None):
        self.dt = float(dt)
        self.mass = ROBOT_MASS
        self.izz = ROBOT_INERTIA
        self.wheel_radius = WHEEL_RADIUS
        self.wheel_inertia = WHEEL_INERTIA
        self.current_limit = IQ_LIMIT
        self.traction_limit = MU_D_STAT * F_N
        self.max_acceleration = MAX_ACCELERATION
        self.max_velocity = MAX_VELOCITY
        self.max_yaw_rate = MAX_YAW_RATE
        self.disturbance_fn = disturbance_fn
        self.disturbance_forecast = disturbance_forecast
        self.heading_reference_fn = (
            configured_heading_reference() if heading_reference_fn is None else heading_reference_fn
        )

        alpha = np.radians(np.asarray(WHEEL_ALPHA_DEG, dtype=float))
        self.drive = np.vstack((-np.sin(alpha), np.cos(alpha)))
        wheel_radius_vectors = WHEEL_ROBOT_DIST * np.vstack((np.cos(alpha), np.sin(alpha)))
        yaw_levers = wheel_radius_vectors[0] * self.drive[1] - wheel_radius_vectors[1] * self.drive[0]
        self.H = np.vstack((self.drive, yaw_levers))
        # H maps body twist [u, v, omega] to wheel drive speeds and yaw lever arms.
        # It is the matrix form of the old wheel-geometry force/moment sums.
        self.body_mass = np.diag([self.mass, self.mass, self.izz])
        # Rolling couples wheel spin inertia back into the body: M_eff = M_body
        # + (J_w/R^2) H H^T. This keeps wheel inertia in the coupled dynamics.
        self.effective_mass = self.body_mass + (
            self.wheel_inertia / self.wheel_radius**2
        ) * (self.H @ self.H.T)
        self.mass_inverse = np.linalg.inv(self.effective_mass)
        # Map wheel currents to generalized acceleration using motor torque
        # tau_motor=K_t*Iq and the no-slip wheel torque balance.
        self.G = -(MOTOR_KT / self.wheel_radius) * self.mass_inverse @ self.H

        facet_angles = 2.0 * np.pi * np.arange(8) / 8.0
        self.facets = np.column_stack((np.cos(facet_angles), np.sin(facet_angles)))
        self.acceleration_limit_poly = self.max_acceleration * np.cos(np.pi / 8.0)
        self.velocity_limit_poly = self.max_velocity * np.cos(np.pi / 8.0)

        self.state = self.reference_state(0.0) if initial_state is None else np.asarray(initial_state, dtype=float).copy()
        if self.state.shape != (6,):
            raise ValueError("initial_state must be [x_world, y_world, phi, vx_world, vy_world, omega]")
        self.initial_state = self.state.copy()
        self.index = 0
        self.history = []

    def heading_ref(self, time_s):
        position, velocity, _ = trajectory_reference(time_s)
        return float(self.heading_reference_fn(time_s, position, velocity))

    def yaw_rate_ref(self, time_s):
        eps = max(1e-4, self.dt * 0.25)
        return wrap_angle(self.heading_ref(time_s + eps) - self.heading_ref(time_s - eps)) / (2.0 * eps)

    def reference_state(self, time_s):
        position, velocity, _ = trajectory_reference(time_s)
        return np.r_[position, self.heading_ref(time_s), velocity, self.yaw_rate_ref(time_s)]

    @staticmethod
    def coriolis(twist_body, mass):
        u, v, omega = twist_body
        return np.array([-mass * omega * v, mass * omega * u, 0.0])

    def physical_dynamics(self, state_world, current, wrench_body):
        """Apply motor/contact/disturbance forces and return world-state rates.

        Velocity is rotated to body axes for the wheel model, the body force
        balance is solved there, and translational acceleration is rotated back.
        """
        phi = state_world[2]
        velocity_body = world_to_body(state_world[3:5], phi)
        twist_body = np.r_[velocity_body, state_world[5]]
        coriolis = self.coriolis(twist_body, self.mass)
        generalized_acceleration = self.G @ current + self.mass_inverse @ (wrench_body - coriolis)
        acceleration_body = np.array([
            generalized_acceleration[0] - state_world[5] * velocity_body[1],
            generalized_acceleration[1] + state_world[5] * velocity_body[0],
        ])
        acceleration_world = body_to_world(acceleration_body, phi)
        # Required drive traction is the contact force needed to satisfy rolling
        # and wheel torque balance; MPC limits it by static friction capacity.
        traction = (
            -(MOTOR_KT / self.wheel_radius) * current
            - (self.wheel_inertia / self.wheel_radius**2) * (self.H.T @ generalized_acceleration)
        )
        derivative_world = np.r_[state_world[3:5], state_world[5], acceleration_world, generalized_acceleration[2]]
        return derivative_world, traction, generalized_acceleration, velocity_body

    def wheel_dynamics(self, current, traction, generalized_acceleration, twist_body):
        """Apply no-slip kinematics and signed load torque from actuator.py/simulation.py."""
        theta_dot = -(self.H.T @ twist_body) / self.wheel_radius
        theta_ddot = -(self.H.T @ generalized_acceleration) / self.wheel_radius
        motor_torque = MOTOR_KT * current
        signed_load_torque = self.wheel_radius * traction
        torque_residual = self.wheel_inertia * theta_ddot - (motor_torque + signed_load_torque)
        return theta_dot, theta_ddot, motor_torque, signed_load_torque, torque_residual

    def discrete_map(self, state, current, wrench_body):
        derivative, _, _, _ = self.physical_dynamics(state, current, wrench_body)
        next_state = state + self.dt * derivative
        next_state[:2] += 0.5 * self.dt**2 * derivative[3:5]
        next_state[2] += 0.5 * self.dt**2 * derivative[5]
        return next_state

    def _linearize(self, function, state0, current0, wrench0):
        """Finite-difference A/B/E around the nominal state, current, and wrench.

        The physical dynamics remain nonlinear, but this local affine model is
        what each convex MPC subproblem predicts over its short horizon.
        """
        value0 = np.asarray(function(state0, current0, wrench0), dtype=float)
        jacobians = [np.zeros((value0.size, 6)),
                     np.zeros((value0.size, N_WHEELS)),
                     np.zeros((value0.size, 3))]
        epsilon = 1e-5
        for block, width in enumerate((6, N_WHEELS, 3)):
            for column in range(width):
                plus = [state0.copy(), current0.copy(), wrench0.copy()]
                minus = [state0.copy(), current0.copy(), wrench0.copy()]
                plus[block][column] += epsilon
                minus[block][column] -= epsilon
                jacobians[block][:, column] = (function(*plus) - function(*minus)) / (2.0 * epsilon)
        affine = value0 - jacobians[0] @ state0 - jacobians[1] @ current0 - jacobians[2] @ wrench0
        return (*jacobians, affine)

    def linear_model(self, time_index=None, wrench_body=None):
        """Locally linearize world-state dynamics, body wrench, acceleration, and traction."""
        time_index = self.index if time_index is None else int(time_index)
        time_s = time_index * self.dt
        wrench0 = np.zeros(3) if wrench_body is None else np.asarray(wrench_body, dtype=float)
        # The axial friction law is nonlinear in wheel-frame slip. For the
        # current MPC stage, linearize at the measured plant state so a large
        # tracking error is not extrapolated from a zero-slip reference.
        state0 = (
            self.state.copy() if time_index == self.index
            else self.reference_state(time_s)
        )
        eps = max(1e-4, self.dt * 0.25)
        twist_ref_dot = (
            self.reference_state(time_s + eps)[3:6]
            - self.reference_state(time_s - eps)[3:6]
        ) / (2.0 * eps)
        velocity_body = world_to_body(state0[3:5], state0[2])
        qdot_ref = np.r_[twist_ref_dot[:2], twist_ref_dot[2]]
        qdot_ref[:2] += np.array([-state0[5] * velocity_body[1], state0[5] * velocity_body[0]])
        current0 = np.linalg.pinv(self.G) @ (
            qdot_ref - self.mass_inverse @ (wrench0 - self.coriolis(np.r_[velocity_body, state0[5]], self.mass))
        )

        A, B, E, c = self._linearize(self.discrete_map, state0, current0, wrench0)
        acceleration_map = self._linearize(
            lambda state, current, wrench: self.physical_dynamics(state, current, wrench)[0][3:5],
            state0, current0, wrench0,
        )
        traction_map = self._linearize(
            lambda state, current, wrench: self.physical_dynamics(state, current, wrench)[1],
            state0, current0, wrench0,
        )
        return {
            "A": A, "B": B, "E": E, "c": c,
            "acceleration_x": acceleration_map[0], "acceleration_u": acceleration_map[1],
            "acceleration_w": acceleration_map[2], "acceleration_c": acceleration_map[3],
            "traction_x": traction_map[0], "traction_u": traction_map[1],
            "traction_w": traction_map[2], "traction_c": traction_map[3],
            "nominal_state": state0, "nominal_current": current0,
        }

    def candidate_feasible(self, scale, request, state, wrench, model):
        current = scale * request
        if model is None:
            # Check PID's candidate against the actual nonlinear equations.
            derivative, traction, _, _ = self.physical_dynamics(state, current, wrench)
            next_state = self.discrete_map(state, current, wrench)
            acceleration = derivative[3:5]
        else:
            next_state = model["A"] @ state + model["B"] @ current + model["E"] @ wrench + model["c"]
            acceleration = model["acceleration_x"] @ state + model["acceleration_u"] @ current + model["acceleration_w"] @ wrench + model["acceleration_c"]
            traction = model["traction_x"] @ state + model["traction_u"] @ current + model["traction_w"] @ wrench + model["traction_c"]
        tolerance = MPC_CONSTRAINT_TOL
        return (
            # Current is an actuator bound, so enforce it exactly even while
            # allowing solver-scale tolerance on the other modeled constraints.
            np.max(np.abs(current)) <= self.current_limit
            and np.max(np.abs(traction)) <= self.traction_limit + tolerance
            and np.max(self.facets @ acceleration) <= self.acceleration_limit_poly + tolerance
            and np.max(self.facets @ next_state[3:5]) <= self.velocity_limit_poly + tolerance
            and abs(next_state[5]) <= self.max_yaw_rate + tolerance
        )

    def largest_feasible_current_scale(self, request, state, wrench, model):
        # A nonzero command may be needed to brake an already saturated state
        # (for example, yaw rate at its limit), so check the requested action
        # before assuming that scaling it toward zero preserves feasibility.
        if self.candidate_feasible(1.0, request, state, wrench, model):
            return 1.0, True
        if not self.candidate_feasible(0.0, request, state, wrench, model):
            return 0.0, False
        low, high = 0.0, 1.0
        for _ in range(40):
            middle = 0.5 * (low + high)
            if self.candidate_feasible(middle, request, state, wrench, model):
                low = middle
            else:
                high = middle
        return low, True

    def project_current_to_feasible(self, request, state, wrench, model):
        """Project an inaccurate MPC first action onto one-step plant limits.

        This is a small convex QP. It also handles boundary states where zero
        current is not a feasible fallback because the robot must actively brake.
        """
        import cvxpy as cp

        current = cp.Variable(N_WHEELS)
        next_state = (
            model["A"] @ state + model["B"] @ current
            + model["E"] @ wrench + model["c"]
        )
        acceleration = (
            model["acceleration_x"] @ state
            + model["acceleration_u"] @ current
            + model["acceleration_w"] @ wrench
            + model["acceleration_c"]
        )
        traction = (
            model["traction_x"] @ state
            + model["traction_u"] @ current
            + model["traction_w"] @ wrench
            + model["traction_c"]
        )
        constraints = [
            current <= self.current_limit,
            -current <= self.current_limit,
            traction <= self.traction_limit,
            -traction <= self.traction_limit,
            self.facets @ acceleration <= self.acceleration_limit_poly,
            self.facets @ next_state[3:5] <= self.velocity_limit_poly,
            next_state[5] <= self.max_yaw_rate,
            -next_state[5] <= self.max_yaw_rate,
        ]
        problem = cp.Problem(cp.Minimize(cp.sum_squares(current - request)), constraints)
        problem.solve(
            solver=cp.OSQP, warm_start=True, verbose=False,
            max_iter=100000, eps_abs=1e-6, eps_rel=1e-6,
        )
        if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE) or current.value is None:
            raise RuntimeError(
                f"No feasible one-step current projection at t={self.index * self.dt:.3f}s "
                f"(QP status: {problem.status})."
            )
        projected = np.asarray(current.value, dtype=float).copy()
        if not self.candidate_feasible(1.0, projected, state, wrench, model):
            raise RuntimeError(
                f"One-step current projection still violates plant limits at "
                f"t={self.index * self.dt:.3f}s (QP status: {problem.status})."
            )
        return projected, problem.status

    def step(self, controller):
        time_s = self.index * self.dt
        wrench = np.asarray(self.disturbance_fn(time_s, self.state.copy()), dtype=float)
        if wrench.shape != (3,):
            raise ValueError("disturbance_fn must return body-frame [Fx, Fy, torque]")
        # PID uses the exact nonlinear plant directly. Finite-difference
        # linearization is needed only by MPC and is costly for this model.
        model = (
            self.linear_model(self.index, wrench)
            if isinstance(controller, ConvexMPCController) else None
        )
        decision = controller.compute(self, wrench, model)
        current = np.asarray(decision["current"], dtype=float)
        before = self.state.copy()
        derivative, traction, generalized_acceleration, velocity_body = self.physical_dynamics(before, current, wrench)
        theta_dot, theta_ddot, motor_torque, signed_load_torque, wheel_residual = self.wheel_dynamics(
            current, traction, generalized_acceleration, np.r_[velocity_body, before[5]]
        )
        after = (
            model["A"] @ before + model["B"] @ current
            + model["E"] @ wrench + model["c"]
            if model is not None else self.discrete_map(before, current, wrench)
        )
        margins = {
            "current_A": self.current_limit - np.abs(current),
            "traction_N": self.traction_limit - np.abs(traction),
            "acceleration_facets": self.acceleration_limit_poly - self.facets @ derivative[3:5],
            "velocity_facets": self.velocity_limit_poly - self.facets @ after[3:5],
            "yaw_rate_rad_s": np.array([self.max_yaw_rate - abs(after[5])]),
        }
        clearance_records = obstacle_clearance_report(after[:2], time_s + self.dt)
        obstacle_margins = np.array(
            [record["margin_m"] for record in clearance_records] or [np.inf], dtype=float
        )
        minimum_margin = min(float(np.min(values)) for values in margins.values())
        minimum_obstacle_margin = float(np.min(obstacle_margins))
        collision = minimum_obstacle_margin < 0.0
        residual = self.body_mass @ generalized_acceleration + self.coriolis(np.r_[velocity_body, before[5]], self.mass) - self.H @ traction - wrench
        position_ref, velocity_ref, acceleration_ref = trajectory_reference(time_s)
        snapshot = {
            "step": self.index, "time": time_s, "dt": self.dt,
            "controller": type(controller).__name__, "state_before": before,
            "state_after": after.copy(),
            "reference": {"position": position_ref, "velocity": velocity_ref,
                          "acceleration": acceleration_ref, "heading": self.heading_ref(time_s),
                          "yaw_rate": self.yaw_rate_ref(time_s)},
            "linear_model": model, "disturbance_wrench_body": wrench.copy(),
            "decision": decision, "current": current.copy(),
            "state_derivative": derivative, "velocity_body": velocity_body,
            "wheel_speed_rolling": theta_dot, "wheel_angular_acceleration": theta_ddot,
            "motor_torque": motor_torque, "signed_load_torque": signed_load_torque,
            "wheel_torque_balance_residual": wheel_residual,
            "required_traction": traction.copy(), "drive_slip": np.zeros(N_WHEELS),
            "force_balance_residual": residual, "constraint_margins": margins,
            "minimum_constraint_margin": minimum_margin, "feasible": minimum_margin >= -MPC_CONSTRAINT_TOL and not collision,
            "obstacle_clearance": clearance_records,
            "minimum_obstacle_clearance_margin": minimum_obstacle_margin,
            "collision": collision,
        }
        self.state = after
        self.index += 1
        self.history.append(snapshot)
        if hasattr(controller, "observe"):
            controller.observe(snapshot)
        return snapshot

    def run(self, controller, steps):
        return [self.step(controller) for _ in range(int(steps))]


In [127]:
class ContinuousHeadingRobotSimulator(RollingOnlyRobotSimulator):
    """Keep the reference angle on the same continuous branch as physical phi."""

    def heading_ref(self, time_s):
        raw_heading = super().heading_ref(time_s)
        if not hasattr(self, "state"):
            return raw_heading
        return float(self.state[2] + wrap_angle(raw_heading - self.state[2]))

In [128]:
class AxialFrictionRobotSimulator(ContinuousHeadingRobotSimulator):
    """Rolling in d with Williams-style passive-roller a-axis friction in the plant."""

    def physical_dynamics(self, state_world, current, wrench_body):
        phi = state_world[2]
        velocity_body = world_to_body(state_world[3:5], phi)
        axial_wrench, slip_a, force_a = axial_friction_wrench_body(
            velocity_body, state_world[5]
        )
        wrench_total = np.asarray(wrench_body, dtype=float) + axial_wrench
        coriolis = self.coriolis(np.r_[velocity_body, state_world[5]], self.mass)
        generalized_acceleration = (
            self.G @ current + self.mass_inverse @ (wrench_total - coriolis)
        )
        acceleration_body = np.array([
            generalized_acceleration[0] - state_world[5] * velocity_body[1],
            generalized_acceleration[1] + state_world[5] * velocity_body[0],
        ])
        acceleration_world = body_to_world(acceleration_body, phi)
        traction_d = (
            -(MOTOR_KT / self.wheel_radius) * current
            - (self.wheel_inertia / self.wheel_radius**2)
            * (self.H.T @ generalized_acceleration)
        )
        derivative_world = np.r_[
            state_world[3:5], state_world[5], acceleration_world,
            generalized_acceleration[2],
        ]
        self._last_axial_wrench = axial_wrench
        self._last_slip_a = slip_a
        self._last_force_a = force_a
        return derivative_world, traction_d, generalized_acceleration, velocity_body

    def step(self, controller):
        snapshot = super().step(controller)
        snapshot["slip_a"] = self._last_slip_a.copy()
        snapshot["force_a"] = self._last_force_a.copy()
        snapshot["axial_wrench_body"] = self._last_axial_wrench.copy()
        snapshot["force_balance_residual"] = (
            snapshot["force_balance_residual"] - self._last_axial_wrench
        )
        return snapshot

In [129]:
class PositionPIDController:
    """Position/velocity/heading PID baseline using the old error-feedback idea.

    Desired acceleration is converted to wheel currents through the local model,
    then scaled if the current or static-traction limit would be exceeded.
    """

    def __init__(self, kp=(5.0, 5.0), ki=(0.45, 0.45), kd=(3.2, 3.2),
                 kp_heading=5.0, ki_heading=0.2, kd_heading=2.0):
        self.kp = np.asarray(kp, dtype=float)
        self.ki = np.asarray(ki, dtype=float)
        self.kd = np.asarray(kd, dtype=float)
        self.kp_heading = float(kp_heading)
        self.ki_heading = float(ki_heading)
        self.kd_heading = float(kd_heading)
        self.integral = np.zeros(3)

    def compute(self, plant, wrench_body, model):
        time_s = plant.index * plant.dt
        position_ref, velocity_ref, acceleration_ref = trajectory_reference(time_s)
        state = plant.state
        position_error = position_ref - state[:2]
        velocity_error = velocity_ref - state[3:5]
        # P position + D velocity + I accumulated position error, plus route
        # acceleration feed-forward. Repulsion is a PID heuristic, not a guarantee.
        acceleration_world = (
            acceleration_ref + self.kp * position_error
            + self.ki * self.integral[:2] + self.kd * velocity_error
            + pid_obstacle_repulsion(state[:2], time_s)
        )
        heading_error = wrap_angle(plant.heading_ref(time_s) - state[2])
        yaw_rate_ref = plant.yaw_rate_ref(time_s)
        eps = max(1e-4, 0.25 * plant.dt)
        yaw_acceleration_ref = (
            plant.yaw_rate_ref(time_s + eps) - plant.yaw_rate_ref(time_s - eps)
        ) / (2.0 * eps)
        yaw_acceleration = (
            yaw_acceleration_ref + self.kp_heading * heading_error
            + self.ki_heading * self.integral[2]
            + self.kd_heading * (yaw_rate_ref - state[5])
        )
        velocity_body = world_to_body(state[3:5], state[2])
        acceleration_body = world_to_body(acceleration_world, state[2])
        omega = state[5]
        generalized_acceleration = np.array([
            acceleration_body[0] + omega * velocity_body[1],
            acceleration_body[1] - omega * velocity_body[0],
            yaw_acceleration,
        ])
        coriolis = plant.coriolis(np.r_[velocity_body, omega], plant.mass)
        requested_current = np.linalg.pinv(plant.G) @ (
            generalized_acceleration
            - plant.mass_inverse @ (wrench_body - coriolis)
        )
        scale, feasible = plant.largest_feasible_current_scale(
            requested_current, state, wrench_body, model
        )
        return {
            "current": scale * requested_current,
            "requested_current": requested_current,
            "tracking_error": np.r_[position_error, heading_error],
            "saturation_scale": scale,
            "feasible_ray": feasible,
            "freeze_integrator": scale < 1.0 - 1e-9,
        }

    def observe(self, snapshot):
        if not snapshot["decision"]["freeze_integrator"]:
            self.integral += snapshot["dt"] * snapshot["decision"]["tracking_error"]
        self.integral = np.clip(self.integral, -1.5, 1.5)


class ConvexMPCController:
    """Receding-horizon convex QP over wheel currents and predicted robot states.

    Position, velocity, and heading references are fixed data at each stage.
    Integral error is predicted controller memory, not an extra plant state.
    """

    def __init__(self, horizon=20, weights=None):
        self.horizon = int(horizon)
        if self.horizon < 1:
            raise ValueError("horizon must be positive")
        self.integral = np.zeros(3)
        self.planned_integrals = None
        self.weights = weights or {
            "position": 60.0, "velocity": 8.0, "heading": 8.0,
            "integral": 0.05, "integral_slack": 1000.0,
            "acceleration": 2.0, "current": 0.02, "slew": 0.15,
            "terminal_position": 120.0, "terminal_velocity": 16.0,
            "terminal_heading": 20.0,
        }

    def _continuous_heading_reference(self, plant, time_s):
        return plant.state[2] + wrap_angle(plant.heading_ref(time_s) - plant.state[2])

    def compute(self, plant, wrench_body, first_model):
        import cvxpy as cp

        requested_horizon = self.horizon
        horizon = self.horizon
        # Decision variables: six physical states at N+1 samples, PID integral
        # memory at N+1 samples, and four wheel currents at N control samples.
        states = cp.Variable((6, horizon + 1))
        integrals = cp.Variable((3, horizon + 1))
        integral_slack = cp.Variable((3, horizon), nonneg=True)
        # Obstacle clearance is softened so an already unsafe state or a changed
        # waypoint cannot make the whole QP infeasible for one short time step.
        # A large penalty makes the optimizer satisfy clearance whenever feasible.
        obstacle_slacks = []
        currents = cp.Variable((N_WHEELS, horizon))
        constraints = [states[:, 0] == plant.state, integrals[:, 0] == self.integral]
        objective = 0

        for offset in range(horizon):
            index = plant.index + offset
            next_time = (index + 1) * plant.dt
            model = first_model if offset == 0 else plant.linear_model(index, np.zeros(3))
            # These are desired targets at this future time, not variables. For a
            # receive-pass event the terminal target is the goal with zero velocity.
            position_ref, velocity_ref, acceleration_ref = trajectory_reference(next_time)
            heading_ref = self._continuous_heading_reference(plant, next_time)
            stage_wrench = wrench_body if offset == 0 else np.zeros(3)
            current = currents[:, offset]
            # The local affine maps express acceleration and required wheel
            # traction as functions of predicted state and candidate currents.
            acceleration = (
                model["acceleration_x"] @ states[:, offset]
                + model["acceleration_u"] @ current
                + model["acceleration_w"] @ stage_wrench
                + model["acceleration_c"]
            )
            traction = (
                model["traction_x"] @ states[:, offset]
                + model["traction_u"] @ current
                + model["traction_w"] @ stage_wrench
                + model["traction_c"]
            )
            # Matrix state update: x[k+1] = A*x[k] + B*Iq[k] + E*w + c.
            # The next two equalities accumulate position and heading error.
            constraints += [
                states[:, offset + 1] == model["A"] @ states[:, offset]
                    + model["B"] @ current + model["E"] @ stage_wrench + model["c"],
                integrals[:2, offset + 1] == integrals[:2, offset]
                    + plant.dt * (position_ref - states[:2, offset]),
                integrals[2, offset + 1] == integrals[2, offset]
                    + plant.dt * (heading_ref - states[2, offset]),
                integrals[:, offset + 1] <= 1.5 + integral_slack[:, offset],
                -integrals[:, offset + 1] <= 1.5 + integral_slack[:, offset],
                current <= plant.current_limit,
                -current <= plant.current_limit,
                # No-slip feasibility: every wheel traction demand must fit
                # within +/- mu_static * normal_load.
                traction <= plant.traction_limit,
                -traction <= plant.traction_limit,
                # Facets make polygonal (therefore linear) bounds for acceleration
                # and speed instead of nonlinear Euclidean-norm constraints.
                plant.facets @ acceleration <= plant.acceleration_limit_poly,
                plant.facets @ states[3:5, offset + 1] <= plant.velocity_limit_poly,
                cp.abs(states[5, offset + 1]) <= plant.max_yaw_rate,
            ]
            # Every obstacle has one preselected safe side. Its separating line
            # is fixed data for this solve, so the added constraint is affine.
            # The obstacle side and separating hyperplane are constants for this
            # QP stage, selected from the nominal reference and predicted obstacle pose.
            for _, obstacle_vertices, obstacle_clearance in obstacle_polygons_at(next_time):
                normal, bound = obstacle_safe_halfspace(
                    position_ref, obstacle_vertices, clearance=obstacle_clearance
                )
                clearance_slack = cp.Variable(nonneg=True)
                obstacle_slacks.append(clearance_slack)
                constraints.append(
                    normal @ states[:2, offset + 1] + clearance_slack >= bound
                )
                # Squared nonnegative slack keeps this a convex QP. It allows a
                # temporary clearance shortfall only when the hard plant limits
                # prevent clearing the obstacle at this sample.
                objective += 1e4 * cp.square(clearance_slack)
            # Quadratic tracking costs target pose/velocity; current and slew
            # penalties regularize effort without changing convexity.
            objective += self.weights["position"] * cp.sum_squares(states[:2, offset + 1] - position_ref)
            objective += self.weights["velocity"] * cp.sum_squares(states[3:5, offset + 1] - velocity_ref)
            objective += self.weights["heading"] * cp.square(states[2, offset + 1] - heading_ref)
            objective += self.weights["integral"] * cp.sum_squares(integrals[:, offset + 1])
            objective += self.weights["integral_slack"] * cp.sum_squares(integral_slack[:, offset])
            objective += self.weights["acceleration"] * cp.sum_squares(acceleration - acceleration_ref)
            objective += self.weights["current"] * cp.sum_squares(current)
            previous = plant.history[-1]["current"] if offset == 0 and plant.history else np.zeros(N_WHEELS)
            if offset > 0:
                previous = currents[:, offset - 1]
            objective += self.weights["slew"] * cp.sum_squares(current - previous)

        terminal_time = (plant.index + horizon) * plant.dt
        terminal_position, terminal_velocity, _ = trajectory_reference(terminal_time)
        terminal_heading = self._continuous_heading_reference(plant, terminal_time)
        objective += self.weights["terminal_position"] * cp.sum_squares(states[:2, -1] - terminal_position)
        objective += self.weights["terminal_velocity"] * cp.sum_squares(states[3:5, -1] - terminal_velocity)
        objective += self.weights["terminal_heading"] * cp.square(states[2, -1] - terminal_heading)
        # Apply only the first current; measure the new state and solve again
        # on the next step (the receding-horizon control loop).
        problem = cp.Problem(cp.Minimize(objective), constraints)
        problem.solve(
            solver=cp.OSQP, warm_start=True, verbose=False,
            max_iter=50000, eps_abs=MPC_CONSTRAINT_TOL, eps_rel=MPC_CONSTRAINT_TOL,
        )
        if problem.status not in (cp.OPTIMAL, cp.OPTIMAL_INACCURATE):
            if horizon > 1:
                shorter = ConvexMPCController(horizon=horizon - 1, weights=self.weights)
                shorter.integral = self.integral.copy()
                decision = shorter.compute(plant, wrench_body, first_model)
                self.planned_integrals = shorter.planned_integrals
                decision["requested_horizon"] = requested_horizon
                return decision
            raise RuntimeError(
                f"MPC QP did not solve at t={plant.index * plant.dt:.3f}s "
                f"after reducing to horizon 1 (solver status: {problem.status}). "
                "A status of 'user_limit' means OSQP reached its iteration limit, "
                "not that the constraints were proven infeasible."
            )
        requested_current = currents.value[:, 0].copy()
        # OSQP can return a numerically inaccurate iterate. Check the first
        # action against the plant limits and scale it toward zero if needed.
        if plant.candidate_feasible(
            1.0, requested_current, plant.state, wrench_body, first_model
        ):
            applied_current = requested_current
            safety_scale = 1.0
            safety_projection_status = None
        elif plant.candidate_feasible(
            0.0, requested_current, plant.state, wrench_body, first_model
        ):
            safety_scale, _ = plant.largest_feasible_current_scale(
                requested_current, plant.state, wrench_body, first_model
            )
            applied_current = safety_scale * requested_current
            safety_projection_status = None
        else:
            # Scaling toward zero fails when the robot must apply braking current
            # to recover a saturated speed/yaw state, so solve a small projection QP.
            applied_current, safety_projection_status = plant.project_current_to_feasible(
                requested_current, plant.state, wrench_body, first_model
            )
            safety_scale = None
        # Enforce the hard actuator cap exactly despite OSQP feasibility tolerance.
        hard_current_limit = min(float(plant.current_limit), float(IQ_LIMIT))
        applied_current = np.clip(applied_current, -hard_current_limit, hard_current_limit)
        self.planned_integrals = integrals.value[:, 1].copy()
        return {
            "current": applied_current,
            "requested_current": requested_current,
            "first_step_safety_scale": safety_scale,
            "first_step_safety_projection_status": safety_projection_status,
            "qp_status": problem.status,
            "qp_objective": float(problem.value),
            "planned_currents": currents.value.copy(),
            "planned_states": states.value.copy(),
            "planned_integrals": integrals.value.copy(),
            "planned_obstacle_slacks": np.asarray(
                [slack.value for slack in obstacle_slacks], dtype=float
            ),
            "effective_horizon": horizon,
            "requested_horizon": requested_horizon,
            "integral_slack": integral_slack.value.copy(),
        }

    def observe(self, snapshot):
        if self.planned_integrals is not None:
            self.integral = np.clip(self.planned_integrals, -1.5, 1.5)


In [ ]:
def make_selected_controller():
    if CONTROLLER_MODE.upper() == "PID":
        return PositionPIDController()
    if CONTROLLER_MODE.upper() == "MPC":
        return ConvexMPCController(horizon=MPC_HORIZON)
    raise ValueError("CONTROLLER_MODE must be 'PID' or 'MPC'")


preview_plant = AxialFrictionRobotSimulator()
preview_controller = make_selected_controller()
for _ in range(3):
    snapshot = preview_plant.step(preview_controller)
    print(
        f"{CONTROLLER_MODE.upper()} k={snapshot['step']} t={snapshot['time']:.2f}s "
        f"p={np.round(snapshot['state_after'][:2], 4)} "
        f"i={np.round(snapshot['current'], 4)} "
        f"min_margin={snapshot['minimum_constraint_margin']:.4f}"
    )
if snapshot["linear_model"] is not None:
    print("Model shapes A/B/E:", snapshot["linear_model"]["A"].shape,
          snapshot["linear_model"]["B"].shape, snapshot["linear_model"]["E"].shape)
else:
    print("PID uses direct nonlinear plant dynamics; skipped MPC linearization.")

GOAL_HOLD_TIME = 1.0  # Keep tracking after arrival so settling is visible.
SIMULATION_TIME = PATH_END_TIME + GOAL_HOLD_TIME
N_STEPS = int(np.ceil(SIMULATION_TIME / DT))
plant = AxialFrictionRobotSimulator()
run_controller = make_selected_controller()
trace = plant.run(run_controller, N_STEPS)
TIME = DT * np.arange(N_STEPS + 1)
state_history = np.vstack((plant.initial_state,
                           np.array([sample["state_after"] for sample in trace])))
position_reference_history = np.array([trajectory_reference(t)[0] for t in TIME])
position_error_history = position_reference_history - state_history[:, :2]
velocity_history = state_history[:, 3:5]
velocity_reference_history = np.array([trajectory_reference(t)[1] for t in TIME])
speed_history = np.linalg.norm(velocity_history, axis=1)

acceleration_history = np.array([
    sample["state_derivative"][3:5] for sample in trace
])
acceleration_reference_history = np.array([
    trajectory_reference(t)[2] for t in TIME[:-1]
])
acceleration_magnitude_history = np.linalg.norm(acceleration_history, axis=1)
current_history = np.array([sample["current"] for sample in trace])
traction_history = np.array([sample["required_traction"] for sample in trace])
torque_residual_history = np.array([sample["wheel_torque_balance_residual"] for sample in trace])
traction_utilization_history = 100.0 * np.abs(traction_history) / F_ROLL_LIMIT
# Axial passive-roller slip/friction is part of this plant independently of the
# drive-direction traction cap, so record it as its own model output.
axial_slip_history = np.array([sample["slip_a"] for sample in trace])
axial_force_history = np.array([sample["force_a"] for sample in trace])

# Recorded issue to revisit: the earlier MPC run hit the obstacle check at k=176 (1.760 s).
# This PID run can also fail the plant safety assertion on collision; PID has no QP status.
infeasible_samples = [sample for sample in trace if not sample["feasible"]]
if infeasible_samples:
    first_bad = infeasible_samples[0]
    raise AssertionError(
        f"{len(infeasible_samples)} infeasible plant samples; first at "
        f"k={first_bad['step']} t={first_bad['time']:.3f}s, "
        f"current={np.max(np.abs(first_bad['current'])):.4f}/{IQ_LIMIT:.4f} A, "
        f"margin={first_bad['minimum_constraint_margin']:.3e}, "
        f"obstacle_collision={first_bad['collision']}"
    )
assert max(np.max(np.abs(sample["force_balance_residual"])) for sample in trace) < 1e-8
assert max(np.max(np.abs(sample["wheel_torque_balance_residual"])) for sample in trace) < 1e-10
assert np.max(np.abs(current_history)) <= IQ_LIMIT + MPC_CONSTRAINT_TOL
assert np.max(np.abs(traction_history)) <= F_ROLL_LIMIT + MPC_CONSTRAINT_TOL
print(f"\nReference transit time: {PATH_END_TIME:.2f} s; "
      f"simulation including goal hold: {SIMULATION_TIME:.2f} s")
print(f"{CONTROLLER_MODE.upper()} run: {N_STEPS} explicit plant steps")
print(f"Position RMS error: {np.sqrt(np.mean(position_error_history**2)):.5f} m per axis")
print(f"Maximum position error: {np.max(np.linalg.norm(position_error_history, axis=1)):.5f} m")
print(f"Peak wheel current: {np.max(np.abs(current_history)):.3f} A / {IQ_LIMIT:.3f} A")
print(f"Peak required drive traction: {np.max(np.abs(traction_history)):.3f} N / {F_ROLL_LIMIT:.3f} N")
print(f"Peak traction utilization: {np.max(traction_utilization_history):.1f}%")
print(f"Peak passive-roller axial slip: {np.max(np.abs(axial_slip_history)):.4f} m/s")
print(f"Peak passive-roller axial friction: {np.max(np.abs(axial_force_history)):.3f} N/wheel")
print(f"Peak speed: {np.max(speed_history):.3f} m/s")
print(f"Peak linear acceleration: {np.max(acceleration_magnitude_history):.3f} m/s^2")
print(f"Peak wheel torque-balance residual: {np.max(np.abs(torque_residual_history)):.2e} N m")

# Explicit IPython display keeps all plots inline in the notebook output.
from IPython.display import HTML, display

# The animated field view is the primary simulation visualization.
# Animated top-down XY view. This silhouette is for display; collision checks
# still use the separate ROBOT_CLEARANCE value in the simulator/MPC.
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Polygon as PolygonPatch
from matplotlib.transforms import Affine2D
from IPython.display import HTML, display

# Use the robot geometry already defined by this project: half of the 158.97 mm
# wheel/contact circle is the simulator's robot radius. The SSL front-cut ratio
# gives the clipped-circle outline; both the board and robot use meters directly.
ROBOT_BODY_RADIUS = WHEEL_ROBOT_DIST
SSL_FRONT_CUT_FRACTION = 0.055 / 0.085
ROBOT_FRONT_CUT_X = SSL_FRONT_CUT_FRACTION * ROBOT_BODY_RADIUS
cut_angle = np.arccos(ROBOT_FRONT_CUT_X / ROBOT_BODY_RADIUS)
arc_angles = np.linspace(cut_angle, 2.0 * np.pi - cut_angle, 100)
robot_outline_local = np.vstack((
    np.column_stack((ROBOT_BODY_RADIUS * np.cos(arc_angles),
                     ROBOT_BODY_RADIUS * np.sin(arc_angles))),
    [ROBOT_FRONT_CUT_X, ROBOT_BODY_RADIUS * np.sin(cut_angle)],
))
front_half_width = ROBOT_BODY_RADIUS * np.sin(cut_angle)
front_edge_local = np.array([
    [ROBOT_FRONT_CUT_X, -front_half_width],
    [ROBOT_FRONT_CUT_X,  front_half_width],
])

anim_fig, (anim_ax, robot_detail_ax) = plt.subplots(
    1, 2, figsize=(12, 7), gridspec_kw={"width_ratios": [4, 1]}
)
# The gray chord is the tempting direct route; both obstacles block it, so the
# controller tracks the curved safe route above them instead.
anim_ax.plot(
    [ROUTE_CONTROL_POINTS[0, 0], ROUTE_CONTROL_POINTS[-1, 0]],
    [ROUTE_CONTROL_POINTS[0, 1], ROUTE_CONTROL_POINTS[-1, 1]],
    color="0.45", linestyle=":", linewidth=1.2, label="direct route (blocked)",
)
anim_ax.plot(position_reference_history[:, 0], position_reference_history[:, 1],
             "k--", linewidth=1.2, label="position target")
trail_line, = anim_ax.plot([], [], color="tab:blue", linewidth=1.8, label="robot path")
reference_marker, = anim_ax.plot([], [], marker="x", color="tab:orange", markersize=8,
                                linestyle="None", label="current target")

# Draw static obstacles in the same world coordinates used by the MPC half-spaces.
for obstacle_index, obstacle_vertices in enumerate(STATIC_OBSTACLES):
    obstacle_patch = PolygonPatch(obstacle_vertices, closed=True, facecolor="0.75",
                                  edgecolor="0.25", alpha=0.65, zorder=1,
                                  label="static obstacle" if obstacle_index == 0 else None)
    anim_ax.add_patch(obstacle_patch)

# Optional moving-obstacle patches follow the supplied prediction callbacks.
dynamic_patches = []
for obstacle_index, obstacle in enumerate(DYNAMIC_OBSTACLES):
    world_vertices = (np.asarray(obstacle["vertices"], dtype=float)
                      + np.asarray(obstacle["position_fn"](0.0), dtype=float))
    obstacle_patch = PolygonPatch(world_vertices, closed=True, facecolor="tab:red",
                                  edgecolor="darkred", alpha=0.45, zorder=2,
                                  label="predicted moving obstacle" if obstacle_index == 0 else None)
    anim_ax.add_patch(obstacle_patch)
    dynamic_patches.append(obstacle_patch)

# Rotate the clipped circle by phi, then translate it to [x, y]. Its flat front
# edge makes heading changes (including pass/shot turns) visible in the field view.
robot_patch = PolygonPatch(robot_outline_local, closed=True, facecolor="tab:blue",
                           edgecolor="black", linewidth=1.5, zorder=5, label="robot")
anim_ax.add_patch(robot_patch)
front_edge_line, = anim_ax.plot([], [], color="crimson", linewidth=2.5, zorder=6)
# A separate close-up keeps the real-size body legible without enlarging it on
# the board. Its axes stay in meters and the body outline stays at true scale.
robot_detail_ax.set_xlim(-0.32, 0.32)
robot_detail_ax.set_ylim(-0.32, 0.32)
robot_detail_ax.set_aspect("equal", adjustable="box")
robot_detail_ax.set_title("Robot detail\n(zoomed; dimensions in meters)")
robot_detail_ax.set_xlabel("body x [m]")
robot_detail_ax.set_ylabel("body y [m]")
robot_detail_ax.grid(True, color="0.88")
robot_detail_ax.set_xticks(np.arange(-0.3, 0.31, 0.1))
robot_detail_ax.set_yticks(np.arange(-0.3, 0.31, 0.1))
robot_detail_patch = PolygonPatch(
    robot_outline_local, closed=True, facecolor="tab:blue",
    edgecolor="black", linewidth=1.5, zorder=5,
)
robot_detail_ax.add_patch(robot_detail_patch)
detail_front_edge, = robot_detail_ax.plot([], [], color="crimson", linewidth=2.5, zorder=6)
detail_heading_line, = robot_detail_ax.plot([], [], color="crimson", linewidth=1.5, zorder=7)
# This arrow points along the robot's +x body axis, making heading unambiguous.
from matplotlib.patches import FancyArrowPatch
heading_arrow = FancyArrowPatch((0.0, 0.0), (1.0, 0.0), arrowstyle="-|>",
                                mutation_scale=18, color="crimson", linewidth=2.0,
                                zorder=7, label="heading direction")
anim_ax.add_patch(heading_arrow)
status_text = anim_ax.text(0.02, 0.98, "", transform=anim_ax.transAxes,
                           va="top", ha="left", family="monospace")

# Fixed equal-scale bounds show the actual 4.5 m x 6 m half-field.
anim_ax.set_xlim(-FIELD_LENGTH / 2, FIELD_LENGTH / 2)
anim_ax.set_ylim(-FIELD_WIDTH / 2, FIELD_WIDTH / 2)
from matplotlib.patches import Rectangle
anim_ax.add_patch(Rectangle(
    (-FIELD_LENGTH / 2, -FIELD_WIDTH / 2), FIELD_LENGTH, FIELD_WIDTH,
    fill=False, edgecolor="white", linewidth=2.0, zorder=4,
))
anim_ax.plot([0.0, 0.0], [-FIELD_WIDTH / 2, FIELD_WIDTH / 2],
             color="white", linewidth=1.2, zorder=1)
anim_ax.set_aspect("equal", adjustable="box")
anim_ax.set_xlabel("x [m]")
anim_ax.set_ylabel("y [m]")
anim_ax.set_title("4.5 m x 6 m simulation field: robot pose, route, and obstacles")
anim_ax.set_facecolor("#f1f5f3")
anim_ax.grid(True, color="white", linewidth=1.2)
anim_ax.legend(loc="upper right")

# Downsample long runs so the inline animation stays responsive, preserving endpoints.
frame_indices = np.unique(np.linspace(
    0, len(TIME) - 1, min(240, len(TIME)), dtype=int
))

def update_robot_animation(frame_number):
    sample_index = frame_indices[frame_number]
    x, y, phi = state_history[sample_index, :3]
    robot_patch.set_transform(Affine2D().rotate(phi).translate(x, y) + anim_ax.transData)

    # Draw the front chord in world coordinates as a clear heading marker.
    rotation = np.array([[np.cos(phi), -np.sin(phi)],
                         [np.sin(phi),  np.cos(phi)]])
    front_edge_world = front_edge_local @ rotation.T + np.array([x, y])
    front_edge_line.set_data(front_edge_world[:, 0], front_edge_world[:, 1])
    # Keep the arrow just ahead of the body and point it along heading phi.
    heading_start = np.array([x, y]) + 0.9 * ROBOT_BODY_RADIUS * np.array([np.cos(phi), np.sin(phi)])
    heading_end = np.array([x, y]) + 4.0 * ROBOT_BODY_RADIUS * np.array([np.cos(phi), np.sin(phi)])
    heading_arrow.set_positions(heading_start, heading_end)
    robot_detail_patch.set_transform(Affine2D().rotate(phi) + robot_detail_ax.transData)
    detail_front_world = front_edge_local @ rotation.T
    detail_front_edge.set_data(detail_front_world[:, 0], detail_front_world[:, 1])
    detail_heading_line.set_data(
        [0.9 * ROBOT_BODY_RADIUS * np.cos(phi), 4.0 * ROBOT_BODY_RADIUS * np.cos(phi)],
        [0.9 * ROBOT_BODY_RADIUS * np.sin(phi), 4.0 * ROBOT_BODY_RADIUS * np.sin(phi)],
    )
    trail_line.set_data(state_history[:sample_index + 1, 0],
                        state_history[:sample_index + 1, 1])
    reference_marker.set_data([position_reference_history[sample_index, 0]],
                              [position_reference_history[sample_index, 1]])

    # Moving obstacle positions are evaluated at the displayed simulation time.
    for patch, obstacle in zip(dynamic_patches, DYNAMIC_OBSTACLES):
        translation = np.asarray(obstacle["position_fn"](TIME[sample_index]), dtype=float)
        patch.set_xy(np.asarray(obstacle["vertices"], dtype=float) + translation)

    speed = np.linalg.norm(state_history[sample_index, 3:5])
    status_text.set_text(
        f"t = {TIME[sample_index]:.2f} s\n"
        f"heading = {np.degrees(phi):.1f} deg\n"
        f"speed = {speed:.2f} m/s"
    )
    return [robot_patch, front_edge_line, heading_arrow, robot_detail_patch,
            detail_front_edge, detail_heading_line, trail_line, reference_marker, status_text,
            *dynamic_patches]

robot_animation = FuncAnimation(
    anim_fig, update_robot_animation, frames=len(frame_indices),
    interval=max(20, int(1000 * SIMULATION_TIME / max(1, len(frame_indices)))),
    blit=False, repeat=False,
)
plt.close(anim_fig)  # Show the HTML animation instead of a duplicate static frame.
display(HTML(robot_animation.to_jshtml()))

# Scalar plots below are secondary diagnostics and remain inline as well.
figure, axes = plt.subplots(4, 2, figsize=(13, 16))
axes[0, 0].plot(position_reference_history[:, 0], position_reference_history[:, 1], "k--", label="reference")
axes[0, 0].plot(state_history[:, 0], state_history[:, 1], label=CONTROLLER_MODE.upper())
axes[0, 0].set_aspect("equal", adjustable="box")
axes[0, 0].set_xlabel("x [m]")
axes[0, 0].set_ylabel("y [m]")
axes[0, 0].set_title("Position trajectory")
axes[0, 0].legend()
axes[0, 0].grid(True)

axes[0, 1].plot(TIME, position_error_history[:, 0], label="x error")
axes[0, 1].plot(TIME, position_error_history[:, 1], label="y error")
axes[0, 1].set_ylabel("Position error [m]")
axes[0, 1].set_title("Tracking error")
axes[0, 1].legend()
axes[0, 1].grid(True)

for wheel in range(N_WHEELS):
    axes[1, 0].plot(TIME[:-1], current_history[:, wheel], label=f"wheel {wheel + 1}")
axes[1, 0].set_ylim(-IQ_LIMIT, IQ_LIMIT)
axes[1, 0].axhline(IQ_LIMIT, color="black", linestyle=":")
axes[1, 0].axhline(-IQ_LIMIT, color="black", linestyle=":")
axes[1, 0].set_ylabel("Current [A]")
axes[1, 0].set_title(f"Wheel current (limit +/-{IQ_LIMIT:g} A)")
axes[1, 0].legend(ncol=2)
axes[1, 0].grid(True)

for wheel in range(N_WHEELS):
    axes[1, 1].plot(TIME[:-1], traction_history[:, wheel], label=f"wheel {wheel + 1}")
axes[1, 1].set_ylim(-F_ROLL_LIMIT, F_ROLL_LIMIT)
axes[1, 1].axhline(F_ROLL_LIMIT, color="black", linestyle=":", label="static traction limit")
axes[1, 1].axhline(-F_ROLL_LIMIT, color="black", linestyle=":")
axes[1, 1].set_ylabel("Required drive traction [N]")
axes[1, 1].set_title(f"Required traction (limit +/-{F_ROLL_LIMIT:.2f} N/wheel)")
axes[1, 1].legend(ncol=2)
axes[1, 1].grid(True)

for wheel in range(N_WHEELS):
    axes[2, 0].plot(TIME[:-1], traction_utilization_history[:, wheel], label=f"wheel {wheel + 1}")
axes[2, 0].axhline(100.0, color="black", linestyle=":", label="static-friction limit")
axes[2, 0].set_ylim(0.0, 105.0)
axes[2, 0].set_xlabel("Time [s]")
axes[2, 0].set_ylabel("Static traction capacity [%]")
axes[2, 0].set_title("No-slip margin")
axes[2, 0].legend(ncol=2)
axes[2, 0].grid(True)

axial_force_ax = axes[2, 1].twinx()
for wheel in range(N_WHEELS):
    axes[2, 1].plot(TIME[:-1], axial_slip_history[:, wheel],
                    label=f"wheel {wheel + 1} slip")
    axial_force_ax.plot(TIME[:-1], axial_force_history[:, wheel], linestyle="--",
                        label=f"wheel {wheel + 1} force")
axes[2, 1].axhline(0.0, color="0.35", linewidth=0.8)
axes[2, 1].set_xlabel("Time [s]")
axes[2, 1].set_ylabel("Axial roller slip [m/s]")
axial_force_ax.set_ylabel("Axial friction force [N]")
axes[2, 1].set_title("Passive-roller axial slip and friction")
slip_handles, slip_labels = axes[2, 1].get_legend_handles_labels()
force_handles, force_labels = axial_force_ax.get_legend_handles_labels()
axes[2, 1].legend(slip_handles + force_handles, slip_labels + force_labels,
                  ncol=2, fontsize=7)
axes[2, 1].grid(True)

# Linear velocity
axes[3, 0].plot(TIME, velocity_history[:, 0], label="vx actual")
axes[3, 0].plot(TIME, velocity_reference_history[:, 0], "--", label="vx reference")
axes[3, 0].plot(TIME, velocity_history[:, 1], label="vy actual")
axes[3, 0].plot(TIME, velocity_reference_history[:, 1], "--", label="vy reference")
axes[3, 0].plot(TIME, speed_history, label="speed magnitude")
axes[3, 0].set_xlabel("Time [s]")
axes[3, 0].set_ylabel("Velocity [m/s]")
axes[3, 0].set_title("Linear velocity")
axes[3, 0].legend(ncol=2)
axes[3, 0].grid(True)

# Linear acceleration
axes[3, 1].plot(TIME[:-1], acceleration_history[:, 0], label="ax actual")
axes[3, 1].plot(TIME[:-1], acceleration_reference_history[:, 0], "--", label="ax reference")
axes[3, 1].plot(TIME[:-1], acceleration_history[:, 1], label="ay actual")
axes[3, 1].plot(TIME[:-1], acceleration_reference_history[:, 1], "--", label="ay reference")
axes[3, 1].plot(TIME[:-1], acceleration_magnitude_history, label="|a| actual")
axes[3, 1].axhline(MAX_ACCELERATION, color="black", linestyle=":", label="acceleration limit")
axes[3, 1].set_xlabel("Time [s]")
axes[3, 1].set_ylabel("Acceleration [m/s²]")
axes[3, 1].set_title("Linear acceleration")
axes[3, 1].legend(ncol=2)
axes[3, 1].grid(True)

figure.tight_layout()
display(figure)  # Render in the notebook output instead of opening a GUI window.
plt.close(figure)




## 5. Convex MPC: One Decision at a Time

`ConvexMPCController.compute()` solves a finite-horizon QP over the same augmented position/integral state and returns the full predicted state/current sequence. `simulator.step(controller)` applies only the first current and returns the matrices, prediction, and constraint margins for inspection; call it again for the next receding-horizon decision.

The objective is a sum of squared position, velocity, integral-state, acceleration-reference, current, and current-slew errors. The dynamics are affine and all current, static-traction, acceleration, speed, and integral bounds are linear, so the optimization is a convex QP. References are time-indexed across the horizon, not an endpoint target.

In [ ]:
# Keep the standalone MPC demonstration dormant while validating the plant with PID.
if CONTROLLER_MODE.upper() == "MPC":
    mpc_plant = RollingOnlyRobotSimulator()
    mpc_controller = ConvexMPCController(horizon=20)
    mpc_snapshot = mpc_plant.step(mpc_controller)

    assert mpc_snapshot["decision"]["qp_status"] in ("optimal", "optimal_inaccurate")
    assert mpc_snapshot["decision"]["planned_states"].shape == (6, 21)
    assert mpc_snapshot["decision"]["planned_currents"].shape == (N_WHEELS, 20)
    assert mpc_snapshot["feasible"]

    print("QP status:", mpc_snapshot["decision"]["qp_status"])
    print("Objective:", round(mpc_snapshot["decision"]["qp_objective"], 6))
    print("First current decision [A]:", np.round(mpc_snapshot["current"], 5))
    print("Predicted state/current shapes:",
          mpc_snapshot["decision"]["planned_states"].shape,
          mpc_snapshot["decision"]["planned_currents"].shape)
    print("Minimum plant constraint margin:", round(mpc_snapshot["minimum_constraint_margin"], 6))
else:
    print("Standalone MPC demo skipped because CONTROLLER_MODE is PID.")